# Fetch StatsBomb Open Event Data

Fetches all available open event data from StatsBomb via `statsbombpy`.
Events are saved to `data/events.parquet` for downstream causal analysis
of early yellow cards on defensive behavior.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import pandas as pd
from tqdm.notebook import tqdm
from statsbombpy import sb

## 1. Available competitions

In [ ]:
competitions = sb.competitions()
print(f"{len(competitions)} competition-seasons available")
competitions[["competition_id", "season_id", "competition_name", "season_name", "competition_gender"]].sort_values(
    ["competition_name", "season_name"]
).reset_index(drop=True)

## 2. Fetch all matches

In [ ]:
match_frames = []

for _, row in tqdm(competitions.iterrows(), total=len(competitions), desc="Competitions"):
    try:
        m = sb.matches(competition_id=row["competition_id"], season_id=row["season_id"])
        m["competition_name"] = row["competition_name"]
        m["season_name"] = row["season_name"]
        m["competition_gender"] = row["competition_gender"]
        match_frames.append(m)
    except Exception as e:
        print(f"  Skipped {row['competition_name']} {row['season_name']}: {e}")

matches = pd.concat(match_frames, ignore_index=True)
print(f"{len(matches)} matches across {matches['competition_name'].nunique()} competitions")
matches.head(3)

## 3. Fetch all events

We drop `shot_freeze_frame` and `tactics` columns (nested objects, not needed for basic event analysis).
Events are streamed match-by-match and concatenated into a single DataFrame.

In [ ]:
# Columns to drop — large nested structures not needed for causal analysis
DROP_COLS = {"shot_freeze_frame", "tactics", "related_events"}

event_frames = []
failed_matches = []

match_ids = matches["match_id"].tolist()

for match_id in tqdm(match_ids, desc="Matches"):
    try:
        ev = sb.events(match_id=match_id)
        ev.drop(columns=[c for c in DROP_COLS if c in ev.columns], inplace=True)
        event_frames.append(ev)
    except Exception as e:
        failed_matches.append((match_id, str(e)))

if failed_matches:
    print(f"Failed to fetch {len(failed_matches)} matches:")
    for mid, err in failed_matches[:10]:
        print(f"  match_id={mid}: {err}")

events = pd.concat(event_frames, ignore_index=True)
print(f"\n{len(events):,} events from {events['match_id'].nunique():,} matches")

## 4. Quick sanity check

In [ ]:
print("Event types:")
print(events["type"].value_counts().to_string())

print("\nFoul cards (yellow card events):")
cards = events[events["foul_committed_card"].notna()]
print(cards["foul_committed_card"].value_counts())
print(f"\nMatches with at least one yellow card: {cards['match_id'].nunique():,}")

## 5. Fetch player nationality from lineups

In [ ]:
nationality_frames = []
failed_lineups = []

for match_id in tqdm(match_ids, desc="Lineups"):
    try:
        lineups = sb.lineups(match_id=match_id)
        for team_df in lineups.values():
            nationality_frames.append(
                team_df[["player_id", "country"]].rename(columns={"country": "nationality"})
            )
    except Exception as e:
        failed_lineups.append((match_id, str(e)))

if failed_lineups:
    print(f"Failed: {len(failed_lineups)} matches")

# A player's nationality is constant; keep one row per player
nationality = (
    pd.concat(nationality_frames, ignore_index=True)
    .dropna(subset=["player_id", "nationality"])
    .drop_duplicates(subset="player_id")
)

print(f"{len(nationality):,} unique players with nationality")
print(f"Top nationalities:\n{nationality['nationality'].value_counts().head(10).to_string()}")

In [ ]:
nationality.to_csv(data_dir / "nationality.csv", index=False)
print(f"Saved {len(nationality):,} players  →  data/nationality.csv")

## 6. Save to disk

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

def _series_to_arrow(s):
    """Convert a pandas Series to an Arrow array without triggering
    pandas extension-type registration. Falls back to string on type conflicts."""
    arr = s.to_numpy(dtype=object, na_value=None)
    try:
        return pa.array(arr, from_pandas=True)
    except (pa.lib.ArrowInvalid, pa.lib.ArrowTypeError):
        return pa.array([str(v) if v is not None else None for v in arr], type=pa.string())

def save_parquet(df, path):
    table = pa.table({col: _series_to_arrow(df[col]) for col in df.columns})
    pq.write_table(table, str(path))

save_parquet(matches, data_dir / "matches.parquet")
save_parquet(events,  data_dir / "events.parquet")

print(f"Saved {len(matches):,} matches  →  data/matches.parquet")
print(f"Saved {len(events):,} events   →  data/events.parquet")
print(f"events.parquet size: {(data_dir / 'events.parquet').stat().st_size / 1e6:.1f} MB")